In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-5"

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=None, stop_sequences=None, tools=None):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if temperature is not None:
        params["temperature"] = temperature

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if tools:
        params["tools"] = tools

    return client.messages.create(**params)


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

In [3]:
# Implementation of the text editor tool
#
# The text editor is an Anthropic-defined *client* tool: Anthropic publishes the
# schema and trains Claude on it, but every command runs in our code. The
# `text_editor_20250728` version (Claude 4 and later) supports four commands:
# view, create, str_replace and insert.
#
# Two safety measures the docs recommend for any implementation:
# - Path validation: the model-supplied `path` is untrusted, so every operation
#   is confined to base_dir to prevent directory traversal (e.g. "../../etc").
# - Backups: a timestamped copy is saved before any modification, so a bad edit
#   is always recoverable.
import os
import shutil
from typing import List, Optional


class TextEditorTool:
    def __init__(self, base_dir: str = "", backup_dir: str = ""):
        self.base_dir = os.path.abspath(base_dir or os.getcwd())
        self.backup_dir = os.path.abspath(
            backup_dir or os.path.join(self.base_dir, ".backups")
        )
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path: str) -> str:
        abs_path = os.path.abspath(os.path.join(self.base_dir, file_path))
        if os.path.commonpath([abs_path, self.base_dir]) != self.base_dir:
            raise ValueError(
                f"Access denied: Path '{file_path}' is outside the allowed directory"
            )
        return abs_path

    def _backup_file(self, abs_path: str) -> None:
        if os.path.exists(abs_path):
            file_name = os.path.basename(abs_path)
            backup_path = os.path.join(
                self.backup_dir, f"{file_name}.{os.path.getmtime(abs_path):.0f}"
            )
            shutil.copy2(abs_path, backup_path)

    def view(self, file_path: str, view_range: Optional[List[int]] = None) -> str:
        abs_path = self._validate_path(file_path)

        if os.path.isdir(abs_path):
            return "\n".join(os.listdir(abs_path))

        if not os.path.exists(abs_path):
            raise FileNotFoundError(f"File not found: {file_path}")

        with open(abs_path, "r", encoding="utf-8") as f:
            lines = f.read().split("\n")

        start, end = 1, len(lines)
        if view_range:
            start, end = view_range
            if end == -1:
                end = len(lines)

        # Line numbers let Claude target view_range and insert_line precisely.
        return "\n".join(
            f"{i}: {line}" for i, line in enumerate(lines[start - 1 : end], start)
        )

    def create(self, file_path: str, file_text: str) -> str:
        abs_path = self._validate_path(file_path)

        # Overwriting an existing file is allowed, but back it up first.
        self._backup_file(abs_path)
        os.makedirs(os.path.dirname(abs_path), exist_ok=True)

        with open(abs_path, "w", encoding="utf-8") as f:
            f.write(file_text)

        return f"Successfully created {file_path}"

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        abs_path = self._validate_path(file_path)

        if not os.path.exists(abs_path):
            raise FileNotFoundError(f"File not found: {file_path}")

        with open(abs_path, "r", encoding="utf-8") as f:
            content = f.read()

        # The contract requires exactly one match: erroring on 0 or >1 matches
        # forces Claude to provide enough context for an unambiguous edit.
        match_count = content.count(old_str)
        if match_count == 0:
            raise ValueError(
                "No match found for replacement. Please check your text and try again."
            )
        if match_count > 1:
            raise ValueError(
                f"Found {match_count} matches for replacement text. "
                "Please provide more context to make a unique match."
            )

        self._backup_file(abs_path)
        with open(abs_path, "w", encoding="utf-8") as f:
            f.write(content.replace(old_str, new_str))

        return "Successfully replaced text at exactly one location."

    def insert(self, file_path: str, insert_line: int, insert_text: str) -> str:
        abs_path = self._validate_path(file_path)

        if not os.path.exists(abs_path):
            raise FileNotFoundError(f"File not found: {file_path}")

        with open(abs_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        # insert_line is the line *after which* to insert; 0 means the beginning.
        if not 0 <= insert_line <= len(lines):
            raise IndexError(
                f"Line number {insert_line} is out of range. "
                f"File has {len(lines)} lines."
            )

        if not insert_text.endswith("\n"):
            insert_text += "\n"

        self._backup_file(abs_path)
        lines.insert(insert_line, insert_text)
        with open(abs_path, "w", encoding="utf-8") as f:
            f.writelines(lines)

        return f"Successfully inserted text after line {insert_line}"

In [4]:
# Process tool call requests
#
# Claude edits files inside a sandbox directory (editor_workspace/) rather than
# the repository root. The dispatch matches the tool's declared name,
# `str_replace_based_edit_tool`, and the command names / input fields of
# text_editor_20250728: view, create, str_replace, insert.
import json

text_editor_tool = TextEditorTool(base_dir="editor_workspace")


def run_tool(tool_name, tool_input):
    if tool_name != "str_replace_based_edit_tool":
        raise Exception(f"Unknown tool name: {tool_name}")

    command = tool_input["command"]
    if command == "view":
        return text_editor_tool.view(tool_input["path"], tool_input.get("view_range"))
    elif command == "create":
        return text_editor_tool.create(tool_input["path"], tool_input["file_text"])
    elif command == "str_replace":
        return text_editor_tool.str_replace(
            tool_input["path"], tool_input["old_str"], tool_input["new_str"]
        )
    elif command == "insert":
        return text_editor_tool.insert(
            tool_input["path"], tool_input["insert_line"], tool_input["insert_text"]
        )
    else:
        raise Exception(f"Unknown text editor command: {command}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [5]:
# Tool declaration
#
# Anthropic-defined tools are declared by `type` and `name` only — the input
# schema is built into the model, so no input_schema is sent.
# `text_editor_20250728` + `str_replace_based_edit_tool` is the pair for
# Claude 4 and later models (earlier models used text_editor_20250124).
text_editor_schema = {
    "type": "text_editor_20250728",
    "name": "str_replace_based_edit_tool",
}

In [6]:
# Run the conversation in a loop until the model doesn't ask for a tool use.
# The system prompt tells Claude where the workspace root is — without it,
# the model wastes turns probing absolute paths (/repo, /workspace, ...) that
# the path validation rejects.
def run_conversation(messages):
    while True:
        response = chat(
            messages,
            system=(
                "You can edit files with the text editor tool. Use paths "
                "relative to the workspace root, e.g. 'notes.txt' or "
                "'src/app.py'. Do not use absolute paths."
            ),
            tools=[text_editor_schema],
        )

        add_assistant_message(messages, response)

        text = text_from_message(response)
        if text:
            print(text)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [7]:
messages = []

add_user_message(
    messages,
    "Create a file called fizzbuzz.py with a fizzbuzz(n) function, "
    "then review the file and add a module-level docstring at the top.",
)

run_conversation(messages)

I'll create the fizzbuzz.py file first, then review and add a docstring.


Now let me review the file before adding the docstring:


The file looks good — a straightforward `fizzbuzz(n)` implementation. Now I'll add a module-level docstring at the top.


Let's verify the final result:


Done! I've created `fizzbuzz.py` with:

1. **A `fizzbuzz(n)` function** that iterates from 1 to `n`, printing:
   - `"FizzBuzz"` for multiples of both 3 and 5
   - `"Fizz"` for multiples of 3
   - `"Buzz"` for multiples of 5
   - the number itself otherwise

2. **A module-level docstring** at the top explaining the purpose of the file and what the function does.

The final file is clean, well-documented, and ready to use.


[{'role': 'user',
  'content': 'Create a file called fizzbuzz.py with a fizzbuzz(n) function, then review the file and add a module-level docstring at the top.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll create the fizzbuzz.py file first, then review and add a docstring.", type='text'),
   ToolUseBlock(id='toolu_015RW5FknuqkyVXeyBUeg4sX', caller=DirectCaller(type='direct'), input={'command': 'create', 'path': 'fizzbuzz.py', 'file_text': 'def fizzbuzz(n):\n    for i in range(1, n + 1):\n        if i % 15 == 0:\n            print("FizzBuzz")\n        elif i % 3 == 0:\n            print("Fizz")\n        elif i % 5 == 0:\n            print("Buzz")\n        else:\n            print(i)\n'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_015RW5FknuqkyVXeyBUeg4sX',
    'content': '"Successfully created fizzbuzz.py"',
    'is_error': False}]},
 {'role': 'assistant',
  'con